# Mutual Fund Analytics - Day 1: Data Ingestion & API Integration

Welcome to the interactive notebook for **Day 1 Tasks**. In this session, we will load the official Bluestock datasets, inspect their properties, validate their identification keys (AMFI codes), evaluate data quality issues, and pull real-time fund prices from the public API.

### Step 1: Environment & System Path Setup
We configure our path so that the notebook can access scripts in our `src/` folder, and then import standard data science tools: `pandas`, `numpy`, `matplotlib`, and `seaborn`.

In [ ]:
import sys
from pathlib import Path

# Find the folder where this notebook is, and get its parent folder (the project root)
notebook_dir = Path.cwd()
project_root = notebook_dir.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import json

# Import our dynamic path helper
from src.utils import get_data_dir

# Configure a clean aesthetic for our plots
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
print("Setup complete!")

### Step 2: Automatically Detect and Load CSV Datasets
We scan the `data/raw` folder dynamically to find all CSV files, load them into a dictionary of pandas DataFrames, and print their sizes (shapes).

In [ ]:
raw_dir = get_data_dir("raw")
csv_files = sorted(raw_dir.glob("*.csv"))

print(f"Detected {len(csv_files)} CSV files in data/raw:\n")
loaded_datasets = {}

for f in csv_files:
    # We'll skip mock_returns.csv to focus on the 10 official files
    if f.name == "mock_returns.csv":
        continue
    df = pd.read_csv(f)
    loaded_datasets[f.name] = df
    print(f"Loaded '{f.name}' | Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")

### Step 3: Print Summaries and Details
Let's write a loop to print the column data types, first 5 rows (head), and basic description for each dataset. Let's inspect the `01_fund_master.csv` file first as it contains our fund dictionary.

In [ ]:
# Inspecting 01_fund_master.csv
fund_master = loaded_datasets.get("01_fund_master.csv")
print("=== Column Details for Fund Master ===")
print(fund_master.info())
print("\n=== First 3 Rows ===")
display(fund_master.head(3))
print("\n=== Statistical Summary ===")
display(fund_master.describe(include='all'))

### Step 4: Validate AMFI Codes
AMFI codes act as identifier keys for mutual funds. Let's check that all codes in the master list are positive, numeric, unique, and present. We will also check if other datasets have referencing integrity.

In [ ]:
master_codes = fund_master['amfi_code'].dropna().astype(int)
master_set = set(master_codes)

print(f"Total AMFI codes in master database: {len(master_codes)}")
print(f"Unique AMFI codes in master database: {len(master_set)}")
print(f"Missing / null code count: {fund_master['amfi_code'].isnull().sum()}")

print("\n=== Cross-Reference Integrity Check ===")
for name, df in loaded_datasets.items():
    if name == "01_fund_master.csv":
        continue
    if "amfi_code" in df.columns:
        df_codes = df['amfi_code'].dropna().astype(int)
        orphans = [c for c in df_codes if c not in master_set]
        orphan_count = len(set(orphans))
        print(f"- '{name}': {len(df_codes)} records, {df_codes.nunique()} unique codes, {orphan_count} orphaned codes not in master list.")

### Step 5: Evaluate Data Quality Problems & Plot Missing Values
We count duplicate rows and missing data points, and plot the percentage of missing values per column across our datasets to identify cleaning requirements.

In [ ]:
missing_percentages = {}
for name, df in loaded_datasets.items():
    null_pct = (df.isnull().sum() / len(df)) * 100
    for col, pct in null_pct.items():
        if pct > 0:
            missing_percentages[f"{name.split('_')[1].split('.')[0]}: {col}"] = pct

if missing_percentages:
    ser = pd.Series(missing_percentages).sort_values(ascending=False)
    sns.barplot(x=ser.values, y=ser.index, palette="viridis")
    plt.title("Percentage of Missing Values by Column across Datasets", fontsize=13, fontweight='bold')
    plt.xlabel("Percentage Missing (%)")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found in any datasets! Awesome.")

### Step 6: Fetch Live NAV Data using the mfapi.in API
Now we fetch current pricing data for 5 major mutual fund schemes: 
- HDFC Top 100 Fund (`125497`)
- SBI Bluechip Fund (`119551`)
- ICICI Pru Bluechip Fund (`120503`)
- Nippon India Large Cap Fund (`118632`)
- Axis Bluechip Fund (`119092`)

We save the responses as JSON, and print the live parameters.

In [ ]:
target_codes = [125497, 119551, 120503, 118632, 119092]
summary_rows = []

for code in target_codes:
    url = f"https://api.mfapi.in/mf/{code}"
    try:
        res = requests.get(url, timeout=10)
        if res.status_code == 200:
            data_json = res.json()
            
            # Write to raw json
            json_path = raw_dir / f"live_nav_{code}.json"
            with open(json_path, "w", encoding="utf-8") as jf:
                json.dump(data_json, jf, indent=2)
                
            meta = data_json.get("meta", {})
            nav_data = data_json.get("data", [])
            
            name = meta.get("scheme_name", "Unknown")
            latest_nav = nav_data[0].get("nav", "N/A") if nav_data else "N/A"
            latest_date = nav_data[0].get("date", "N/A") if nav_data else "N/A"
            
            summary_rows.append({
                "AMFI Code": code,
                "Scheme Name": name,
                "Live Date": latest_date,
                "Live NAV": latest_nav
            })
            print(f"Success: {code} -> {name[:40]}... | NAV: {latest_nav} ({latest_date})")
        else:
            print(f"Failed: {code} -> HTTP status code {res.status_code}")
    except Exception as e:
        print(f"Failed: {code} -> Error: {e}")

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(raw_dir / "live_nav_summary.csv", index=False)
print("\n=== Live Consolidated Summary Table ===")
display(summary_df)